In [53]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
# 1. Chargement des données (Utilise tes noms de fichiers sauvegardés)
X_train = pd.read_csv("../features/global_features/X_train.csv")
y_train = pd.read_csv("../features/global_features/y_train.csv")
X_test = pd.read_csv("../features/global_features/X_test.csv")

In [54]:
from sklearn.decomposition import PCA
import pandas as pd

# 1. Identifier les colonnes CLIP (en supposant qu'elles contiennent 'clip' dans leur nom)
# Si elles n'ont pas de nom spécifique, adapte l'indexation (ex: X.iloc[:, :512])
clip_cols = [c for c in X_train.iloc[:,23:535].columns]
other_cols = [c for c in X_train.columns if c not in clip_cols]

print(f"Nombre de features CLIP détectées : {len(clip_cols)}")
print(f"Nombre d'autres features : {len(other_cols)}")

# 2. Initialiser la PCA
# n_components=32 est un bon point de départ pour 512 dimensions sur 1600 vidéos
n_components = 5
pca = PCA(n_components=n_components, random_state=42)

# 3. Fit & Transform sur le TRAIN
clip_pca_train = pca.fit_transform(X_train[clip_cols])

# 4. Transform uniquement sur le TEST (on n'utilise pas fit ici !)
clip_pca_test = pca.transform(X_test[clip_cols])

# 5. Conversion en DataFrame pour reconstruction
clip_pca_train_df = pd.DataFrame(
    clip_pca_train, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_train.index
)
clip_pca_test_df = pd.DataFrame(
    clip_pca_test, 
    columns=[f'pca_clip_{i}' for i in range(n_components)],
    index=X_test.index
)

# 6. Assemblage final : Autres features + Composantes PCA
X_train = pd.concat([X_train[other_cols], clip_pca_train_df], axis=1)
X_test = pd.concat([X_test[other_cols], clip_pca_test_df], axis=1)

print(f"Nouvelle forme de X_train : {X_train.shape}")
print(f"Variance expliquée cumulée : {pca.explained_variance_ratio_.sum():.2%}")

Nombre de features CLIP détectées : 512
Nombre d'autres features : 84
Nouvelle forme de X_train : (1348, 89)
Variance expliquée cumulée : 28.40%


In [55]:
# 2. Préparation
y_train = y_train.values.flatten()
test_ids = X_test['video_id']
X = X_train.drop(columns=['video_id'], errors='ignore')
X_test_final = X_test.drop(columns=['video_id'], errors='ignore')

# 3. Configuration du K-Fold
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Listes pour stocker les scores de validation et les prédictions finales
oof_preds = np.zeros(len(X)) # Out-of-fold predictions
test_preds = np.zeros(len(X_test_final))
cv_scores = []

In [56]:
X_train = pd.DataFrame(X_train)
y_train = pd.DataFrame(y_train)
X_test = pd.DataFrame(X_test)

In [57]:

# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds   = np.zeros(len(X_train))
test_preds  = np.zeros(len(X_test))
feature_cols = X_train.columns
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*50}")
print(f"KFold CV — {5} folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test) / 5
    feature_imp        += model.feature_importances_ / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

oof_rmse = np.sqrt(mean_squared_error(y_train, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")


KFold CV — 5 folds
[200]	valid_0's rmse: 1.12606
  Fold 1 | Best iter:  149 | RMSE: 1.1209
[200]	valid_0's rmse: 1.25529
[400]	valid_0's rmse: 1.25723
  Fold 2 | Best iter:  305 | RMSE: 1.2524
[200]	valid_0's rmse: 1.27355
  Fold 3 | Best iter:  126 | RMSE: 1.2673
[200]	valid_0's rmse: 1.21339
  Fold 4 | Best iter:  244 | RMSE: 1.2098
[200]	valid_0's rmse: 1.23755
  Fold 5 | Best iter:  228 | RMSE: 1.2345

OOF RMSE global : 1.2181


In [58]:
# =========================
# 5) Feature importance top 20
# =========================
fi_df = pd.DataFrame({"feature": feature_cols, "importance": feature_imp})
fi_df = fi_df.sort_values("importance", ascending=False).head(20)
print("\nTop 20 features:")
print(fi_df.to_string(index=False))


Top 20 features:
        feature  importance
       video_id       377.2
     pca_clip_3       313.2
     pca_clip_2       293.0
     pca_clip_1       237.0
     pca_clip_0       236.4
hashtag_density       224.4
       text_len       197.6
     pca_clip_4       178.8
  emoji_density       174.8
           W2_x       169.4
   f2_sharpness       164.0
           W1_x       163.2
  f1_brightness       160.0
           W3_x       159.4
   f1_sharpness       148.6
  f1_saturation       147.8
  f2_saturation       136.8
           B1_x       132.0
     hook_shake       131.4
           R3_x       128.8


In [59]:
# On isole l'ID pour la soumission finale (très important !)
test_ids = X_test['video_id'].copy()

# On définit les features en supprimant video_id
# errors='ignore' permet de ne pas planter si la colonne est déjà absente
X_train = X_train.drop(columns=['video_id'], errors='ignore')
X_test = X_test.drop(columns=['video_id'], errors='ignore')

# On s'assure que y_train est un array 1D pour les calculs de metrics
# y_train doit être la colonne 'score' uniquement
y_train_values = y_train.values.flatten()

In [60]:

# =========================
# 3) LightGBM params
# =========================
lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.03,
    "num_leaves":       127,
    "max_depth":        -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "n_jobs":           -1,
    "verbose":          -1,
    "random_state":     42,
}
# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds   = np.zeros(len(X_train))
test_preds  = np.zeros(len(X_test))
feature_cols = X_train.columns # Maintenant sans video_id
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*50}")
print(f"KFold CV — 5 folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train_values)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_values[train_idx], y_train_values[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test) / 5
    feature_imp        += model.feature_importances_ / 5

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

# Utilisation de y_train_values pour le calcul final
oof_rmse = np.sqrt(mean_squared_error(y_train_values, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")


KFold CV — 5 folds
[200]	valid_0's rmse: 1.15048
  Fold 1 | Best iter:  233 | RMSE: 1.1469
[200]	valid_0's rmse: 1.2642
  Fold 2 | Best iter:  194 | RMSE: 1.2626
[200]	valid_0's rmse: 1.27731
  Fold 3 | Best iter:  102 | RMSE: 1.2671
[200]	valid_0's rmse: 1.2367
  Fold 4 | Best iter:  143 | RMSE: 1.2290
[200]	valid_0's rmse: 1.27539
  Fold 5 | Best iter:  126 | RMSE: 1.2711

OOF RMSE global : 1.2362


In [61]:
# =========================
# 5) Feature importance top 20
# =========================
fi_df = pd.DataFrame({"feature": feature_cols, "importance": feature_imp})
fi_df = fi_df.sort_values("importance", ascending=False).head(20)
print("\nTop 20 features:")
print(fi_df.to_string(index=False))


Top 20 features:
        feature  importance
     pca_clip_3       258.4
     pca_clip_2       249.2
     pca_clip_1       214.4
     pca_clip_0       188.6
       text_len       152.2
hashtag_density       149.4
     pca_clip_4       141.8
   release_year       140.6
  f1_brightness       131.4
   f1_sharpness       131.0
           W1_x       124.6
  f1_saturation       116.2
           W2_x       115.2
   f2_sharpness       114.6
  f2_brightness       113.4
  emoji_density       113.0
           W3_x       112.4
  f2_saturation       106.2
           R3_x       106.0
     hook_shake       104.4


In [35]:
# =========================
# 6) Submission
# =========================
submission = pd.DataFrame({
    'ID': test_ids,
    'popularity': test_preds # Vérifie le nom de colonne attendu
})


# Vérif
assert submission["ID"].isna().sum() == 0, "IDs manquants dans la submission!"
assert submission["popularity"].isna().sum() == 0, "Prédictions NaN dans la submission!"

out_path = "submission_lgbm2.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅ Submission sauvegardée : {out_path}")
print(f"   Shape : {submission.shape}")
print(f"\nAperçu:")
print(submission.head(10).to_string(index=False))
print(f"\nStats popularity prédit:")
print(submission["popularity"].describe().round(4))


✅ Submission sauvegardée : submission_lgbm2.csv
   Shape : (338, 2)

Aperçu:
                 ID  popularity
6764678537346616582    8.022807
6768860777525824773    7.367752
6797039273053850886    9.577828
6800982671657880837    7.213163
6864458828012899589    7.988190
6878219367390268673    6.603545
6879407335484214530    8.575324
6891319697220898050    8.613792
6904609211431357698    7.267014
6915045654997912833    7.008828

Stats popularity prédit:
count    338.0000
mean       7.6973
std        1.1004
min        5.3997
25%        6.9694
50%        7.5061
75%        8.1168
max       10.9250
Name: popularity, dtype: float64
